# Bài học 09 - Mẫu thiết kế siêu nhận thức


## Thiết lập

Notebook này trình bày mẫu thiết kế Metacognition bằng cách sử dụng Microsoft Agent Framework.

**Yêu cầu:**
- Triển khai Azure OpenAI được cấu hình thông qua biến môi trường
- Azure CLI đã được xác thực (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Siêu nhận thức là gì?

Siêu nhận thức là **tư duy về tư duy**. Trong bối cảnh các tác nhân AI, điều đó có nghĩa là xây dựng những tác nhân có thể:

- **Tự suy ngẫm** về kết quả và quá trình suy luận của chính chúng
- **Phát hiện lỗi** và phục hồi một cách khéo léo thay vì thất bại trong im lặng
- **Đánh giá** xem phản hồi của chúng có đầy đủ và hữu ích hay không
- **Thích nghi** chiến lược khi cách tiếp cận ban đầu không hiệu quả (ví dụ, chuyển sang hệ thống dự phòng)

Một tác nhân siêu nhận thức không chỉ trả lời câu hỏi — nó theo dõi hiệu suất của chính nó và điều chỉnh linh hoạt.


## Công cụ chính và dự phòng

Một mẫu siêu nhận thức phổ biến là **chiến lược dự phòng**. The agent tries a primary tool first; if it fails (e.g., a 404 error), the agent recognizes the failure and transparently switches to a backup tool.

This mirrors real-world systems where primary services may be unavailable and agents must self-diagnose the issue before choosing an alternative path.

Dưới đây chúng tôi định nghĩa hai công cụ tra cứu chuyến bay:
- **Chính** — bao gồm Paris, Tokyo và Barcelona
- **Dự phòng** — bao gồm Berlin, Sydney và Thành phố New York


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Tác nhân tự phản ánh với khôi phục lỗi

Tác nhân dưới đây được hướng dẫn thử hệ thống bay chính trước, nhận diện lỗi, và minh bạch chuyển sang hệ thống dự phòng. Sau mỗi phản hồi nó tự đánh giá ngắn gọn liệu nó đã trả lời đầy đủ câu hỏi của người dùng hay chưa.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mẫu Tự Đánh Giá

Một khía cạnh khác của siêu nhận thức là **tự đánh giá**: một tác nhân riêng (hoặc cùng một tác nhân trong lần duyệt thứ hai) xem xét một phản hồi để đánh giá tính đầy đủ, chính xác và hữu ích.

Dưới đây chúng tôi tạo một tác nhân `ResponseEvaluator` chấm điểm các phản hồi của đại lý du lịch theo ba tiêu chí.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Tóm tắt

Trong bài học này bạn đã học cách xây dựng **tác nhân siêu nhận thức** sử dụng Microsoft Agent Framework:

- **Tự phản tư**: Các tác nhân giám sát quá trình suy luận của chính họ và truyền đạt một cách minh bạch những gì đã xảy ra.
- **Phục hồi lỗi với phương án dự phòng**: Mô hình công cụ chính + dự phòng nơi tác nhân phát hiện các thất bại (ví dụ, lỗi 404) và tự động thử nguồn thay thế.
- **Tự đánh giá**: Một tác nhân đánh giá riêng biệt chấm điểm các phản hồi về tính đầy đủ, chính xác và tính hữu ích.

Những mẫu này làm cho các tác nhân bền bỉ hơn, minh bạch hơn và đáng tin cậy hơn — những đặc tính then chốt cho việc triển khai trong môi trường sản xuất.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI Co-op Translator (https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực để đảm bảo tính chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc sai sót. Văn bản gốc bằng ngôn ngữ gốc của tài liệu nên được xem là nguồn tham chiếu chính thức. Đối với các thông tin quan trọng, nên sử dụng bản dịch chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ sự hiểu nhầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
